RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [80]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
chunks = process_all_pdfs("../data")

Found 6 PDF files to process

Processing: bone fracture classes.pdf
  ✓ Loaded 9 pages

Processing: bone fracture.pdf
  ✓ Loaded 6 pages

Processing: bones.pdf
  ✓ Loaded 8 pages

Processing: Bone_Fracture_Classification_in_X-Ray_Using_Deep_Learning_Models.pdf
  ✓ Loaded 10 pages

Processing: Chapter9.pdf
  ✓ Loaded 104 pages

Processing: example1.pdf
  ✓ Loaded 6 pages

Total documents loaded: 143


In [81]:

chunks


[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-03-31T11:58:33+08:00', 'author': 'Jung Han Hwang, Jae Won Seo, Jeong Ho Kim, Suyoung Park, Young Jae Kim and Kwang Gi Kim', 'keywords': 'deep vein thrombosis; computed tomography; radiomics; deep learning; machine learning', 'moddate': '2022-03-31T08:30:26+02:00', 'subject': 'In this study, we aimed to investigate quantitative differences in performance in terms of comparing the automated classification of deep vein thrombosis (DVT) using two categories of artificial intelligence algorithms: deep learning based on convolutional neural networks (CNNs) and conventional machine learning. We retrospectively enrolled 659 participants (DVT patients, 282; normal controls, 377) who were evaluated using contrast-enhanced lower extremity computed tomography (CT) venography. Conventional machine learning consists of logistic regression (LR), support vector machines (SVM), random forests (RF),

Embedding And vectorStoreDB

In [68]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [82]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8584.98it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\manna\AppData\Local\Temp\ipykernel_10980\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [70]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 477


In [83]:
## convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts

['/gid00030/gid00035/gid00032/gid00030/gid00038/gid00001/gid00033/gid00042/gid00045/gid00001\n/gid00048/gid00043/gid00031/gid00028/gid00047/gid00032/gid00046\nCitation: Hwang, J.H.; Seo, J.W.; Kim,\nJ.H.; Park, S.; Kim, Y.J.; Kim, K.G.\nComparison between Deep Learning\nand Conventional Machine Learning\nin Classifying Iliofemoral Deep\nVenous Thrombosis upon CT\nVenography.Diagnostics 2022, 12, 274.\nhttps://doi.org/10.3390/\ndiagnostics12020274\nAcademic Editor: Md\nMohaimenul Islam\nReceived: 11 December 2021\nAccepted: 19 January 2022\nPublished: 21 January 2022\nPublisher’s Note: MDPI stays neutral\nwith regard to jurisdictional claims in\npublished maps and institutional afﬁl-\niations.\nCopyright: © 2022 by the authors.\nLicensee MDPI, Basel, Switzerland.\nThis article is an open access article\ndistributed under the terms and\nconditions of the Creative Commons\nAttribution (CC BY) license (https://\ncreativecommons.org/licenses/by/\n4.0/).\ndiagnostics \nArticle\nComparison be

In [84]:
## convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate embeddings for the documents
embeddings=embedding_manager.generate_embeddings(texts)

## store the documents and embeddings in the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 143 texts...


Batches: 100%|██████████| 5/5 [00:01<00:00,  2.83it/s]


Generated embeddings with shape: (143, 384)
Adding 143 documents to vector store...
Successfully added 143 documents to vector store
Total documents in collection: 763


Retriever Pipeline From VectorStore

In [85]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [86]:
rag_retriever

In [87]:
rag_retriever.retrieve("what is   Iliofemoral Deep Venous Thrombosis" )

Retrieving documents for query: 'what is   Iliofemoral Deep Venous Thrombosis'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 125.02it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bd96be83_16',
  'content': '/gid00030/gid00035/gid00032/gid00030/gid00038/gid00001/gid00033/gid00042/gid00045/gid00001\n/gid00048/gid00043/gid00031/gid00028/gid00047/gid00032/gid00046\nCitation: Hwang, J.H.; Seo, J.W.; Kim,\nJ.H.; Park, S.; Kim, Y.J.; Kim, K.G.\nComparison between Deep Learning\nand Conventional Machine Learning\nin Classifying Iliofemoral Deep\nVenous Thrombosis upon CT\nVenography.Diagnostics 2022, 12, 274.\nhttps://doi.org/10.3390/\ndiagnostics12020274\nAcademic Editor: Md\nMohaimenul Islam\nReceived: 11 December 2021\nAccepted: 19 January 2022\nPublished: 21 January 2022\nPublisher’s Note: MDPI stays neutral\nwith regard to jurisdictional claims in\npublished maps and institutional afﬁl-\niations.\nCopyright: © 2022 by the authors.\nLicensee MDPI, Basel, Switzerland.\nThis article is an open access article\ndistributed under the terms and\nconditions of the Creative Commons\nAttribution (CC BY) license (https://\ncreativecommons.org/licenses/by/\n4.0/)

In [ ]:
rag_retriever.retrieve("what is Data Structures and  Algorithms Using Java ")

Retrieving documents for query: 'what isData Structures and  Algorithms Using Java '
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 199.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_ce68b5e0_33',
  'content': 'Module 02: Generic Programming\nLecture 03 : Basics of Generic Class\nData Structures and \nAlgorithms Using \nJava\nDebasis Samanta\nDepartment of Computer Science & Engineering, IIT Kharagpur',
  'metadata': {'creator': 'Acrobat PDFMaker 20 for PowerPoint',
   'content_length': 187,
   'creationdate': '2022-04-16T12:02:38+05:30',
   'doc_index': 33,
   'total_pages': 104,
   'source_file': 'Chapter9.pdf',
   'page': 0,
   'page_label': '1',
   'file_type': 'pdf',
   'producer': 'Adobe PDF Library 20.12.80',
   'author': 'Reetirupa',
   'source': '..\\data\\pdf\\Chapter9.pdf',
   'title': '',
   'moddate': '2022-04-16T12:02:38+05:30'},
  'similarity_score': 0.34259510040283203,
  'distance': 0.657404899597168,
  'rank': 1},
 {'id': 'doc_5a88f6de_66',
  'content': 'Module 02: Generic Programming\nLecture 04 : Parameterized Generic Class\nData Structures and \nAlgorithms Using \nJava\nDebasis Samanta\nDepartment of Computer Science & Engineering,


RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

True

Integration Vectordb Context pipeline With LLM output

In [107]:
### Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response

def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [108]:
answer=rag_simple("What is  Classifying Iliofemoral Deep Venous Thrombosis",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is  Classifying Iliofemoral Deep Venous Thrombosis'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 125.02it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Classifying Iliofemoral Deep Venous Thrombosis upon CT Venography.


Enhanced RAG Pipeline Features

In [111]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced(" Bone Fracture Classification in X-Ray Using Deep Learning Models", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: ' Bone Fracture Classification in X-Ray Using Deep Learning Models'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 166.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The paper "Bone Fracture Detection in X-ray Images using Convolutional Neural Network" proposes a deep learning method using X-ray images for early diagnosis of bone disorders and detection of different bone fractures. The convolutional neural network model achieves good performance with a specificity of 89.865%, accuracy of 90% approximately, and area under ROC curve of 0.8088.
Sources: [{'source': 'dae4227fc2337c93b4d949598ced1d3e.pdf', 'page': 0, 'score': 0.6699762940406799, 'preview': 'Bone Fracture Detection in X-ray\nImages using Convolutional Neural\nNetwork\nRinisha Bagaria, Sulochana Wadhwani, A. K. Wadhwani\nMadhav Institute of Technology & Science, Gwalior, India\nCorresponding author: Rinisha Bagaria, Email: rinisha.bagaria19@gmail.com\nIt is critical to design a fracture detect...'}, {'source': 'dae4227fc2337c93b4d949598ced1d3e.pdf', 'page': 0, 'score': 0.6699762940406799, 'preview': 'Bone Fracture Detection in X-ray\nImages using Convolutional Neural\nNetwork\nRin

In [114]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query(" Bone Fracture Classification in X-Ray Using Deep Learning Models", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: ' Bone Fracture Classification in X-Ray Using Deep Learning Models'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 141.21it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Bone Fracture Detection in X-ray
Images using Convolutional Neural
Network
Rinisha Bagaria, Sulochana Wadhwani, A. K. Wadhwani
Madhav Institute of Technology & Science, Gwalior, India
Corresponding author: Rinisha Bagaria, Email: rinisha.bagaria19@gma

il.com
It is critical to design a fracture detection system to offer quick results and reduce
diagnosing errors. Using X-ray images in growing artificial intelligence method-
ologies, especially the deep learning method, has become a practical choice for de-
tecting bone fractures. This research paper suggests a deep learning method using
X-ray images for early diagnosis of bone disorders and also detection of different
bone fractures. The effectiveness of the convolutional neural network model for
classifying bone fractures from normal bones is used. Several significant factors
such as no. of epochs, batch size, type of optimizers and learning rate are consid-
ered to find the best-suited model. Hence, it is found that the convolutional neural
network model has good performance using the specificity of 89.865%, accuracy of
90% approximately, and area under ROC curve of 0.8088.
Keywords: Deep Learning, Medical Image Analysis, X-ray Imaging, Image Process-
ing Techniques, Image Classifi